In [3]:
import mujoco as mj
import mediapy as media
import numpy as np

In [6]:
def generate_mujoco_xml(timestep,o1x,o1y,o2x,o2y,L_p1,L_p2,L_d1,L_d2,link_width,link_height,link_separation,max_torque,q0,
                        inertia_p1,inertia_p2,inertia_d1,inertia_d2,mass_p1,mass_p2,mass_d1,mass_d2,damping_bp1,damping_bp2,damping_p1d1,damping_p2d2):
    xml_template = f"""
<mujoco model="parallel_5_bar_mechanism">
    <compiler angle="degree"/>
    <option gravity="0 0 -9.81" integrator="RK4" timestep="{timestep}"/>
    <asset>
        <texture name="grid" type="2d" builtin="checker" rgb1=".1 .2 .3" rgb2=".2 .3 .4" width="300" height="300"/>
        <material name="grid" texture="grid" texrepeat="8 8" reflectance=".2"/>
     </asset>

    <worldbody>
        <light name="top" pos="0 0 1"/>
        <geom name="worldbody" size="1 1 .01" pos="0 0 -.2" type="plane" material="grid"/>
        <body name="base" pos="0 0 0">
            <body name="proximal1" pos="{o1x} {o1y} {1.5*link_height+1.5*link_separation}" euler="0 0 {np.rad2deg(q0[0])}">
                <inertial pos='0 0 0' mass='{mass_p1}' diaginertia='{inertia_p1} {inertia_p1} {inertia_p1}'/>
                <joint name="joint1" type="hinge" axis="0 0 1" pos="0 0 0"   damping="{damping_bp1}" stiffness="0" ref="{np.rad2deg(q0[0])}"/>
                <geom name="proximal1" type="box" size="{(L_p1)/2} {link_width/2} {link_height/2}" pos="{L_p1/2} 0 0" rgba="1 0 0 1"/>
                <body name="distal1" pos="{L_p1} 0 {-link_height-link_separation}" euler="0 0 {np.rad2deg(q0[1]-q0[0])}">
                    <inertial pos='0 0 0' mass='{mass_d1}' diaginertia='{inertia_d1} {inertia_d1} {inertia_d1}'/>
                    <joint name="joint3" type="hinge" axis="0 0 1" pos="0 0 0"  damping="{damping_p1d1}" stiffness="0" ref="{np.rad2deg(q0[1]-q0[0])}"/>
                    <geom name="distal1" type="box" size="{L_d1/2} {link_width/2} {link_height/2}" pos="{L_d1/2} 0 0" rgba="0 1 0 1"/>
                    <body name="end_effector" pos="{L_d1} 0 0">
					    <geom contype="0" name="end_effector" pos="0 0 0" rgba="0.0 0.8 0.6 1" size=".001" type="sphere"/>
				    </body>
                </body>
            </body>
            <body name="proximal2" pos="{o2x} {o2y} {-1.5*link_height-1.5*link_separation}" euler="0 0 {np.rad2deg(q0[3])}">
                <inertial pos='0 0 0' mass='{mass_p2}' diaginertia='{inertia_p2} {inertia_p2} {inertia_p2}'/>
                <joint name="joint2" type="hinge" axis="0 0 1" pos="0 0 0"  damping="{damping_bp2}" stiffness="0"  ref="{np.rad2deg(q0[3])}"/>
                <geom name="proximal2" type="box" size="{L_p2/2} {(link_width)/2} {link_height/2}" pos="{-L_p2/2} 0 0" rgba="1 0 0 1"/>
                <body name="distal2" pos="{-L_p2} 0 {link_height+link_separation}" euler="0 0 {np.rad2deg(q0[2]-q0[3])}">
                    <inertial pos='0 0 0' mass='{mass_d2}' diaginertia='{inertia_d2} {inertia_d2} {inertia_d2}'/>
                    <joint name="joint4" type="hinge" axis="0 0 1" pos="0 0 0"  damping="{damping_p2d2}" stiffness="0" ref="{np.rad2deg(q0[2]-q0[3])}"/>
                    <geom name="distal2" type="box" size="{L_d2/2} {link_width/2} {link_height/2}" pos="{-L_d2/2} 0 0" rgba="0 1 0 1"/>
                </body>
            </body>
        </body>
        <body name="target" pos=".125 .125 0">
			<joint armature="0" axis="1 0 0" damping="0" limited="true" name="target_x" pos="0 0 0" range="-1 1" ref=".125" stiffness="0" type="slide"/>
			<joint armature="0" axis="0 1 0" damping="0" limited="true" name="target_y" pos="0 0 0" range="-1 1" ref=".125" stiffness="0" type="slide"/>
			<geom conaffinity="0" contype="0" name="target" pos="0 0 0" rgba="0.9 0.2 0.2 1" size="0.01" type="sphere"/>
	    </body>
    </worldbody>
    <equality>
		<connect anchor="{L_d1} 0 0" body1="distal1" body2="distal2" name="equality_constraint"/>
	</equality>
    <sensor>
		<framepos objtype="body" objname="end_effector"/>
		<framelinvel objtype="body" objname="end_effector"/>
	</sensor>
    <actuator>
        <!position joint="joint1" ctrllimited="true"  ctrlrange="0 3.14" kp="3" kv="0.1"/>
        <!position joint="joint2" ctrllimited="true"  ctrlrange="-3.14 0" kp="3" kv="0.1"/>
        <motor joint="joint1" ctrllimited="true"  ctrlrange="-{max_torque} {max_torque}"/>
        <motor joint="joint2" ctrllimited="true"  ctrlrange="-{max_torque} {max_torque}"/>
    </actuator>
</mujoco>
"""
    return xml_template

In [ ]:
timestep=0.01
scale=0.5
o1x=-0.075*scale
o1y=0*scale
o2x=0.075*scale
o2y=0*scale
L_p1=0.2*scale
L_p2=0.2*scale
L_d1=0.4*scale
L_d2=0.4*scale
link_width=0.02
link_height=0.025
link_separation=0.003
max_torque=1.6
q0=[2.356194490192345,
    0.999107158546108,
    -0.999107158546108,
    -2.356194490192345]

inertia_p1=1
inertia_p2=1
inertia_d1=1
inertia_d2=1

mass_p1=1
mass_p2=1
mass_d1=1
mass_d2=1



In [5]:
xml = """
<mujoco>
  <worldbody>
    <light name="top" pos="0 0 1"/>
    <body name="box_and_sphere" euler="0 0 -30">
      <joint name="swing" type="hinge" axis="1 -1 0" pos="-.2 -.2 -.2"/>
      <geom name="red_box" type="box" size=".2 .2 .2" rgba="1 0 0 1"/>
      <geom name="green_sphere" pos=".2 .2 .2" size=".1" rgba="0 1 0 1"/>
    </body>
  </worldbody>
</mujoco>
"""
model = mj.MjModel.from_xml_string(xml)
data = mj.MjData(model)

# enable joint visualization option:
scene_option = mj.MjvOption()
scene_option.flags[mj.mjtVisFlag.mjVIS_JOINT] = True

duration = 3.8  # (seconds)
framerate = 60  # (Hz)

# Simulate and display video.
frames = []
mj.mj_resetData(model, data)
with mj.Renderer(model) as renderer:
  while data.time < duration:
    mujoco.mj_step(model, data)
    if len(frames) < data.time * framerate:
      renderer.update_scene(data, scene_option=scene_option)
      pixels = renderer.render()
      frames.append(pixels)

media.show_video(frames, fps=framerate)

This browser does not support the video tag.
